# 📗 세 검색 경로를 통합하고 멀티홉 근거로 답변합니다

기존 영화 자료로 **전문 검색·벡터 검색·관계 검색을 결합해 조건과 근거가 맞는 답변**을 만듭니다.

## 이번 교안에서 배울 내용

1. `GraphCypherQAChain`으로 관계 후보를 찾고, 전문·벡터 결과와 문서 ID별로 합칩니다.
2. 배우·연도 조건을 지킨 같은 후보에서 PageRank 비중을 비교합니다.
3. 줄거리·양쪽 출연 관계·개봉연도를 근거로 재구성하고 답변의 인용을 대조합니다.

<img src="images/04_lesson02_overview.png" width="1050" alt="세 검색 경로 통합, 배우와 연도 조건 확인 및 PageRank 비교, 멀티홉 근거를 이용한 답변의 세 단계">


## 준비한 영화 자료를 연결합니다

이전 교안에서 사용한 **Neo4j Movies 그래프 171개 개체·253개 관계, 영화 원문 38편,
원문 청크 571개와 768차원 배포 벡터**를 그대로 사용합니다. 영화 관계는 Neo4j 샘플,
줄거리는 버전이 고정된 Wikipedia 원문입니다. 두 출처를 구분해서 인용합니다.

노트북 폴더에서 시작하며, 패키지·DB 설정은 README를 참고하세요.

검색·그래프·모델에 필요한 라이브러리를 불러옵니다.


In [ ]:
# 검색·DB·모델에 필요한 라이브러리를 불러옵니다.
import json
import os
import re
import sys
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.types import RetrieverResultItem
from pydantic import BaseModel, Field


자료 경로를 지정하고 그래프·벡터 파일을 읽는 함수를 불러옵니다.


In [ ]:
# 노트북 기준 경로입니다. 정답은 상위 폴더를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
sys.path.insert(0, str(material_dir.resolve()))
from graph_data import load_graph, read_vectors


`.env`로 Neo4j에 연결하고 임베딩·답변 모델과 Cypher 실행 함수를 준비합니다.


In [ ]:
# 공통 설정을 읽고 교안의 .env를 우선 적용합니다.
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# 모델만 준비합니다. API 호출은 검색·답변 단계에서 합니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 준비한 청크를 자동으로 나누지 않습니다.
)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
address = urlparse(neo4j_uri)
print("Neo4j 연결:", address.hostname, address.port or 7687)


def run_cypher(query, **params):
    """Cypher 결과를 행 딕셔너리의 리스트로 반환합니다."""
    # 인자를 Cypher의 $파라미터로 전달합니다.
    with driver.session() as session:
        # 조회 결과를 딕셔너리 목록으로 반환합니다.
        return [record.data() for record in session.run(query, **params)]


영화 그래프·원문·기존 임베딩을 불러오고 자료 규모를 확인합니다.


In [ ]:
# 영화 그래프·원문과 기존 청크 벡터를 불러옵니다.
movies = load_graph(data_dir / "movies_extraction_packet.json")
chunk_ids, vectors = read_vectors(
    data_dir / "embeddings/movies_complete.npz", movies["chunks"], embedding_model
)
# 검색 결과의 원문 ID로 줄거리를 찾을 사전입니다.
documents_by_id = movies["documents"]
print("개체:", len(movies["nodes"]), "관계:", len(movies["relations"]))
print("문서:", len(documents_by_id), "청크:", len(chunk_ids), "벡터:", vectors.shape)


### 영화 그래프와 원문·청크를 적재합니다

원래 ID·속성·관계·출처를 유지하며 `Movie`, `Person`, `Document`, `Chunk`에 저장합니다.
`RAGEntity`는 영화·인물의 공통 ID 조회와 고유 제약을 위해 추가하는 레이블입니다.

ID 중복 방지 제약을 만들고 영화·인물·관계를 Neo4j에 저장합니다.


In [ ]:
# 개체의 standard_id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT rag_entity_id IF NOT EXISTS FOR (n:RAGEntity) REQUIRE n.standard_id IS UNIQUE")
# 원문 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT document_id IF NOT EXISTS FOR (n:Document) REQUIRE n.id IS UNIQUE")
# 청크 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (n:Chunk) REQUIRE n.id IS UNIQUE")

# 영화·인물을 종류별로 적재합니다.
for entity_type in ["Movie", "Person"]:
    # 현재 종류의 개체만 고릅니다.
    rows = [row for row in movies["nodes"] if row["entity_type"] == entity_type]
    # 현재 종류의 영화·인물 노드를 만들고 속성을 갱신합니다.
    run_cypher(f"""
        // 목록을 개체별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 같은 ID의 노드를 재사용하고, 없으면 생성합니다.
        MERGE (n:RAGEntity {{standard_id: item.standard_id}})
        // 종류 레이블을 추가하고 속성·이름을 갱신합니다.
        SET n:{entity_type}, n += item.properties, n.name = item.name
    """, rows=rows)

# 적재한 노드 사이에 종류별 관계를 연결합니다.
for relation in ["ACTED_IN", "DIRECTED", "PRODUCED", "WROTE", "REVIEWED", "FOLLOWS"]:
    # 현재 유형의 관계만 고릅니다.
    rows = [row for row in movies["relations"] if row["relation"] == relation]
    # 현재 유형의 관계를 연결하고 근거·출처를 저장합니다.
    run_cypher(f"""
        // 목록을 관계별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 관계의 시작·끝 노드를 ID로 찾습니다.
        MATCH (a:RAGEntity {{standard_id: item.subject_id}})
        MATCH (b:RAGEntity {{standard_id: item.object_id}})
        // 같은 유형·방향·근거 ID의 관계가 없으면 만들고, 있으면 재사용합니다.
        MERGE (a)-[r:{relation} {{claim_id: item.claim_id}}]->(b)
        // 관계 속성과 근거·출처를 저장합니다.
        SET r += item.properties, r.evidence = item.evidence,
            r.source_url = item.source_url, r.source_doc_id = item.source_doc_id
    """, rows=rows)


원문·청크·기존 벡터를 영화에 연결하고 전문·벡터 검색 인덱스를 만듭니다.


In [ ]:
# 원문 노드를 먼저 저장합니다.
run_cypher("""
    // 원문 목록을 한 문서당 한 행으로 펼칩니다.
    UNWIND $rows AS item
    // 같은 ID의 원문 노드가 없으면 만들고, 있으면 재사용합니다.
    MERGE (d:Document {id: item.doc_id})
    // 제목·본문·출처 URL을 갱신합니다.
    SET d.title = item.title, d.text = item.text, d.url = item.url
""", rows=list(documents_by_id.values()))

# 원문과 해당 영화를 연결합니다.
run_cypher("""
    // doc_id로 원문을, standard_id로 해당 영화를 찾습니다.
    UNWIND $rows AS item
    MATCH (d:Document {id: item.doc_id})
    MATCH (m:Movie {standard_id: item.standard_id})
    // 원문과 해당 영화를 ABOUT_MOVIE 관계로 연결합니다.
    MERGE (d)-[:ABOUT_MOVIE]->(m)
""", rows=movies["document_links"])

# 청크와 벡터의 순서를 맞춰 전체 저장 목록을 만듭니다.
rows = []
for index, chunk in enumerate(movies["chunks"]):
    # NumPy 벡터를 DB에 전달할 목록으로 바꿉니다.
    rows.append({"id": chunk_ids[index], "doc_id": chunk.metadata["source_doc_id"],
                 "text": chunk.page_content, "embedding": vectors[index].tolist()})

# 전체 청크와 벡터를 한 번에 저장합니다.
run_cypher("""
    UNWIND $rows AS item
    // 원문을 찾고 청크를 ID로 생성·재사용합니다.
    MATCH (d:Document {id: item.doc_id})
    MERGE (c:Chunk {id: item.id})
    // text는 전문 검색, embedding은 벡터 검색, source_id는 원문별 결과 통합에 씁니다.
    SET c.text = item.text, c.source_id = item.doc_id, c.embedding = item.embedding
    // 청크와 원문을 FROM_DOCUMENT 관계로 연결합니다.
    MERGE (c)-[:FROM_DOCUMENT]->(d)
""", rows=rows)

# 벡터 유사도 검색용 인덱스를 만듭니다.
run_cypher("""
    CREATE VECTOR INDEX movie_chunks IF NOT EXISTS
    // Chunk의 embedding 속성을 검색 대상으로 지정합니다.
    FOR (c:Chunk) ON c.embedding
    // 저장된 벡터의 768차원과 코사인 유사도를 설정합니다.
    OPTIONS {indexConfig: {`vector.dimensions`: 768, `vector.similarity_function`: 'cosine'}}
""")
# 벡터 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_chunks', 300)")
# 단어 일치 검색용 전문 인덱스를 만듭니다.
run_cypher("""
    CREATE FULLTEXT INDEX movie_fulltext IF NOT EXISTS
    // Chunk의 text 속성을 검색 대상으로 지정합니다.
    FOR (c:Chunk) ON EACH [c.text]
    // 영어 분석기를 사용하고, 본문 변경을 커밋 시 인덱스에 반영합니다.
    OPTIONS {indexConfig: {`fulltext.analyzer`: 'english', `fulltext.eventually_consistent`: false}}
""")
# 전문 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_fulltext', 300)")
print("기존 문서 임베딩을 적재했습니다. 문서 임베딩 API 호출은 없습니다.")


### 적재된 그래프 전체 구조

청크·원문·영화·인물의 연결과 검색 인덱스를 한눈에 봅니다.

<img src="images/06_loaded_graph_v2.png" width="1000" alt="적재된 영화 그래프: 청크에서 원문과 영화로 연결되며, 인물은 영화 및 다른 인물과 관계를 맺습니다. 검색 인덱스는 청크에 적용됩니다.">


한 영화의 여러 청크가 함께 검색될 수 있습니다. 이번 실습은 영화별로 순위를 합치므로, 검색된 청크 중 영화마다 최고 점수 한 건을 대표로 남깁니다. 청크 자체를 답변 근거로 사용할 때는 같은 영화의 여러 청크를 함께 유지할 수도 있습니다.


In [ ]:
def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # content는 청크 본문, metadata는 청크 ID·원문 ID·유사도를 담는 사전입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
vector_retriever = VectorRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    result_formatter=format_vector,
)


def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 영화별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    result = vector_retriever.search(query_vector=query_vector, top_k=top_k)
    # 영화별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(dict(row))
    return rows


def show_documents(rows):
    """전달받은 모든 검색 결과를 순서대로 조회해 제목과 본문 앞부분을 보여 줍니다."""
    display(pd.DataFrame(rows))
    # 검색 순위를 함께 전달해 DB 조회 후에도 같은 순서를 유지합니다.
    ranked_ids = [{"rank": rank, "source_id": row["source_id"]}
                  for rank, row in enumerate(rows)]
    # 전달받은 원문 전체의 제목과 본문을 Neo4j에서 한 번에 조회합니다.
    documents = run_cypher("""
        UNWIND $rows AS item
        // 검색 결과의 원문 ID로 저장된 문서를 찾습니다.
        MATCH (d:Document {id: item.source_id})
        RETURN d.id AS source_id, d.title AS title, d.text AS text
        // DB의 반환 순서를 원래 검색 순위에 맞춥니다.
        ORDER BY item.rank
    """, rows=ranked_ids)
    for doc in documents:
        print("\n제목:", doc["title"], "| 원문 ID:", doc["source_id"])
        # 긴 원문은 앞 650자만 출력합니다.
        print(doc["text"][:650])


## 1. 줄거리와 관계 조건을 나누어 검색합니다

| 검색 구조 | 사용하는 방법 | 그래프의 역할 |
|---|---|---|
| 교안01: 벡터 진입 뒤 확장 | `VectorCypherRetriever` | 진입 영화에서 출연 관계를 따라 후보를 늘립니다. |
| 교안02: 세 경로 결과 융합 | `GraphCypherQAChain` + 전문·벡터 검색 | 질문의 관계 조건을 만족하는 후보 집합을 별도로 찾습니다. |

두 교안은 같은 데이터로 서로 다른 결합 방식을 구현합니다. 각각 준비 셀부터 독립적으로 시작합니다.
이번에는 학습자가 `semantic_query`·`graph_request`·`filters`를 정하고 검색·재정렬·답변을 셀 순서대로 연결합니다.

**질문:** The Matrix와 배우를 공유하고 1999년 이전 개봉한 다른 영화 중,
뇌에 기밀 데이터를 저장·운반하다 위험에 빠지는 작품은 무엇인가요?

| 질문의 요구 | 사용할 입력 | 확인할 결과 |
|---|---|---|
| 뇌에 기밀 데이터를 운반하는 이야기 | 한글 줄거리 검색어 | 전문 검색·Dense 후보와 원문 |
| The Matrix와 배우 공유 | 출연 관계 두 개 | 공통 배우와 양쪽 관계 ID |
| 1999년 이전 | `released < 1999` | 개봉연도 |

줄거리 검색어와 배우·연도 조건은 직접 나눕니다. 벡터 검색은 한글 질문을 사용하고,
전문 검색은 Jev로 언어를 판단한 뒤, 한글은 구조화된 영어 검색어로 바꾸고 영어는 그대로 사용합니다.

전체 질문을 줄거리 검색어와 배우·연도 조건으로 나누고 질문 벡터를 만듭니다.


In [ ]:
question = "The Matrix와 배우를 공유하고 1999년 이전 개봉한 다른 영화 중, 뇌에 기밀 데이터를 저장·운반하다 위험에 빠지는 작품은 무엇인가요?"
semantic_query = "기밀 정보를 뇌 속 장치에 저장해 운반하는 사람이 위험에 빠지는 영화"
# 질문 벡터를 한 번 만들어 재사용합니다.
query_vector = embedding_model.embed_query(semantic_query)
filters = {"anchor": "The Matrix", "min_year": 0, "max_year": 1998, "min_shared": 1}


### GraphCypherQAChain으로 관계 후보를 가져옵니다

이 체인은 자연어 관계 조건으로 Cypher를 작성하고 DB에서 실행합니다.
`return_direct=True`는 추가 답변 생성 없이 **행 목록**을 돌려주며,
`return_intermediate_steps=True`는 작성한 쿼리를 함께 보여 줍니다.
이번 자료의 영화는 38편이므로 `LIMIT 100`과 `top_k=100`으로 관계 조건 집합의 일부가 임의로 잘리는 것을 피합니다.

이 절에는 APOC가 필요하며, 앞에서 설정한 `NEO4J_USER`·`NEO4J_PASSWORD`로 같은 실습 DB에 연결합니다.
**학습 데이터만 있는 독립 실습 DB**에서 진행합니다.
`allow_dangerous_requests=True`도 권한 제한 기능이 아닙니다. 생성 쿼리의 내용을 반드시 읽으세요.
[GraphCypherQAChain 공식 문서](https://docs.langchain.com/oss/python/integrations/graphs/neo4j_cypher)

한글 관계 질문을 Cypher로 바꿔 조회하는 `GraphCypherQAChain`을 구성합니다.


In [ ]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

# 앞에서 설정한 실습 DB 계정으로 연결합니다.
graph = Neo4jGraph(
    url=neo4j_uri,
    username=os.environ["NEO4J_USER"],
    password=os.environ["NEO4J_PASSWORD"],
    enhanced_schema=False,  # 속성 예시 값 수집을 생략하고 기본 스키마를 사용합니다.
)
# 관계 질문을 지정한 스키마의 조회 Cypher로 바꾸는 규칙을 정합니다.
cypher_prompt = PromptTemplate.from_template("""
다음 스키마만 사용해 조회용 Cypher 하나를 작성하세요.
{schema}
Movie, Person, Document를 사용하세요. ACTED_IN은 인물에서 영화 방향입니다.
Document는 ABOUT_MOVIE로 영화에 연결됩니다. Document.id가 source_id입니다.
영화명이 한글과 영문으로 함께 제시되면 DB의 영문 title을 사용하세요.
공유 배우 수는 count(DISTINCT actor)로 세고 기준 영화 자신은 제외하세요.
기준 영화 제외·개봉연도 조건은 집계 WITH 앞의 WHERE에서 적용하세요.
WITH 뒤에서 사용할 변수는 WITH에 포함하고, 집계한 공유 배우 수 조건은 WITH 뒤에서 적용하세요.
RETURN DISTINCT d.id AS source_id, d.title AS title을 사용하세요.
ORDER BY source_id LIMIT 100으로 일정한 순서로 출력하세요.
줄거리 유사도를 그래프 속성으로 가정하지 마세요. Cypher만 반환하세요.
질문: {question}
""")
# 질문 → Cypher 생성 → DB 조회를 연결합니다.
graph_chain = GraphCypherQAChain.from_llm(
    llm=llm, graph=graph, cypher_prompt=cypher_prompt,
    # 필요한 레이블·관계 유형만 프롬프트에 넣습니다.
    include_types=["Movie", "Person", "Document", "ACTED_IN", "ABOUT_MOVIE"],
    # 조회 결과 최대 100행과 생성 Cypher를 반환합니다.
    return_direct=True, return_intermediate_steps=True, top_k=100,
    # 쿼리 실행에 동의합니다. 쓰기 권한을 제한하지는 않습니다.
    allow_dangerous_requests=True,
)


질문에 개봉연도와 기준 영화 제외 조건을 모두 넣습니다. 출력된 쿼리에서 `ACTED_IN` 방향과 연도 비교를 확인하세요.

한글 관계 질문을 실행하고 생성 Cypher와 후보 원문 ID·제목을 확인합니다.


In [ ]:
graph_request = "매트릭스(The Matrix)와 배우를 최소 한 명 공유하며 1999년 이전에 개봉한 다른 영화를 모두 찾아 주세요. 원문 ID는 source_id, 제목은 title로 반환하세요."
# LLM이 관계 질문을 Cypher로 바꿔 실행하면, 생성된 쿼리를 출력합니다.
graph_result = graph_chain.invoke({"query": graph_request})
print(graph_result["intermediate_steps"][0]["query"])
graph_rows = graph_result["result"]
display(pd.DataFrame(graph_rows))


### 명시한 조건을 파라미터 쿼리와 대조합니다

LLM이 만든 Cypher가 실행됐다는 것만으로 질문 해석이 맞았다고 판단하지 않습니다.
우리가 명시한 조건을 다음 쿼리로 다시 확인하고, 출연 관계 ID까지 보관합니다.
이 쿼리는 뒤에서 최종 후보의 조건을 확인할 때도 재사용합니다.

공유 배우 수·개봉연도로 후보를 찾고 출연 근거를 남기는 Cypher를 작성합니다.


In [ ]:
# 배우·연도 조건과 양쪽 출연 근거를 조회할 쿼리입니다.
eligible_query = """
// 같은 배우를 공유하는 영화 쌍을 찾습니다.
MATCH (anchor:Movie {title: $anchor})<-[r1:ACTED_IN]-(actor:Person)-[r2:ACTED_IN]->(candidate:Movie)
// 연도는 양쪽 경계를 포함하며, 기준 영화 자신은 후보에서 제외합니다.
WHERE candidate <> anchor AND candidate.released >= $min_year AND candidate.released <= $max_year
// 고유 배우 수를 세고 양쪽 출연 관계 ID를 모읍니다.
WITH candidate, count(DISTINCT actor) AS shared_count,
     collect(DISTINCT {actor: actor.name, first_claim: r1.claim_id, second_claim: r2.claim_id}) AS paths
// 집계한 공유 배우 수로 후보를 제한합니다.
WHERE shared_count >= $min_shared
// 영화 후보를 검색 결과와 같은 원문 ID로 연결합니다.
MATCH (d:Document)-[:ABOUT_MOVIE]->(candidate)
RETURN d.id AS source_id, candidate.title AS title, candidate.released AS released,
       candidate.standard_id AS movie_id, shared_count, paths
// 관련성 순위 없이 ID순으로 출력합니다.
ORDER BY source_id
"""


LLM이 찾은 후보와 조건 쿼리의 결과를 원문 ID로 비교합니다.


In [ ]:
# 배우·연도 조건에 맞는 영화와 양쪽 출연 근거를 조회합니다.
eligible_rows = run_cypher(eligible_query, **filters)
allowed_ids = {row["source_id"] for row in eligible_rows}
# 생성 쿼리와 조건 쿼리가 찾은 원문 ID를 비교합니다.
graph_ids = {row["source_id"] for row in graph_rows}
print("조건 쿼리에만 있음:", sorted(allowed_ids - graph_ids))
print("생성 쿼리에만 있음:", sorted(graph_ids - allowed_ids))
display(pd.DataFrame(eligible_rows))


두 차집합이 비어 있으면 이번 질문의 관계 조건 집합이 일치합니다.
차이가 있으면 출력 Cypher의 기준 영화·연도·관계 방향·별칭을 고친 뒤 다시 비교하세요.
이 자료에서 `Johnny Mnemonic`은 관계 조건과 줄거리 조건을 함께 확인할 대상입니다.
`The Devil's Advocate`처럼 관계 조건만 맞는 영화도 있으므로 **관계 조회만으로 답을 확정하지 않습니다.**


### 🖐️ 함께 따라하기: Apollo 13의 관계 조건을 확인합니다

`practice_filters`를 기준 영화 `Apollo 13`, 개봉연도 1999~2000, 공유 배우 최소 1명으로 만드세요. 체인에 같은 조건을 한글로 요청하고 생성 쿼리를 읽으세요. `practice_graph_rows`에 체인의 결과, `practice_eligible`에 파라미터 쿼리 결과를 담아 원문 ID 집합을 비교하세요.

Apollo 13의 배우·연도 조건으로 두 조회 방식의 후보를 비교합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### ✅ 바로 확인 퀴즈

ID 오름차순으로 반환된 관계 결과에서 첫 영화를 가장 관련 있는 영화로 볼 수 있나요?

<details><summary>해설 보기</summary>

ID 정렬은 출력 순서만 고정합니다. 관계 조건을 만족한 문서에는 같은 점수를 주고, 줄거리가 질문에 맞는지는 전문·벡터 검색과 원문으로 판단합니다.

</details>


## 2. 원문 ID로 세 경로의 기여를 합칩니다

교안 01의 검색 결과에 관계 조건을 더해, 같은 원문 ID끼리 순위를 합칩니다.

- **전문 검색:** Jev로 언어를 판단하고, 한글은 구조화된 영어 검색어로 바꿉니다. `SEARCH ... FULLTEXT INDEX`로 조회하며 Lucene의 기본 점수 모형은 BM25입니다.
- **벡터 검색:** 한글 질문 벡터와 청크 벡터의 유사도로 찾습니다.
- **관계 검색:** 공통 배우·개봉연도 조건으로 찾습니다.

전문·벡터 결과는 원문별 최고 점수 청크를 남기고, 세 경로를 `source_id`로 연결합니다.

<img src="images/02_three_routes.png" width="1050" alt="전문·벡터 검색의 순위와 관계 조건 집합을 원문 ID로 합치고 필수 조건을 확인하는 흐름">

### 전문 검색·벡터 검색·그래프를 어디에서 결합할까요?

전문 검색은 `Chunk.text`, 벡터 검색(Dense)은 `Chunk.embedding`을 사용합니다.
두 결과에서 같은 원문의 청크를 합친 뒤, 배우·연도 조건으로 찾은 영화 목록과 원문 ID로 연결합니다.

| 구현 | 결합 단위와 방식 | 선택할 때 |
|---|---|---|
| `HybridRetriever` | 같은 노드의 전문 검색·벡터 점수를 정규화해 결합 | 두 검색 경로의 노드 후보를 간결하게 조회할 때 |
| `HybridCypherRetriever` | 하이브리드 후보에서 추가 Cypher 실행 | 결합한 진입 노드에서 관계·근거를 확장할 때 |
| 이번 교안의 명시적 융합 | 청크를 원문 ID로 합치고 문서 순위·관계 기여를 각각 보관 | 중복 처리·경로별 기여·필수 조건을 직접 제어할 때 |

패키지 1.18.0의 `HybridRetriever` 기본 `naive`는 각 경로의 최고 점수로 정규화한 뒤 노드별 최댓값을 선택합니다.
`linear`는 `alpha`로 두 정규화 점수를 가중 합합니다. 둘 다 아래에서 구현하는 RRF와 계산 방식이 다릅니다.
[공식 HybridRetriever 구현](https://neo4j.com/docs/neo4j-graphrag-python/current/_modules/neo4j_graphrag/retrievers/hybrid.html)

Neo4j의 공식 하이브리드 예제도 전문 검색·벡터 검색에 가중 RRF를 적용합니다.
이번에는 **문서 단위 중복 제거와 별도 관계 조건 집합**까지 드러내기 위해 결합 코드를 직접 읽습니다.
[공식 하이브리드 검색 가이드](https://neo4j.com/developer/genai-ecosystem/hybrid-search/)

두 교안은 독립적으로 시작할 수 있습니다. 아래에서 `movie_fulltext`의 `ONLINE` 상태와 `english` 설정을 확인하세요.
`IF NOT EXISTS`는 기존 인덱스 설정을 변경하지 않습니다.

전문 인덱스의 준비 상태와 영어 분석기 설정을 확인합니다.


In [ ]:
# 인덱스 상태와 실제 분석기 설정을 확인합니다.
display(pd.DataFrame(run_cypher("""
    SHOW FULLTEXT INDEXES YIELD name, state, populationPercent, options
    WHERE name = 'movie_fulltext'
    RETURN name, state, populationPercent, options
""")))


언어에 맞는 검색어로 전문 검색하고, 원문별 최고 점수 청크를 반환하는 함수를 만듭니다.


In [ ]:
# Jev가 한글·영어 중 질문의 언어를 선택하도록 설정합니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영어 영화명 등이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


# 영어 검색어를 문자열 목록으로 받습니다.
class EnglishKeywords(BaseModel):
    """영어 원문 검색에 사용할 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 영어 단어 4~8개")


# 한글 질문에서 영어 검색어 목록을 만드는 체인입니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "한글 영화 줄거리 질문을 영어 원문 검색용 핵심어 4~8개로 바꾸세요. "
               "질문의 핵심 사건·대상을 나타내는 단어를 골라 keywords의 각 항목에 영어 단어 하나씩 담으세요. "
               "영화 정답이나 배우를 추측해 추가하지 마세요. 질문에 없는 고유명사는 넣지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(EnglishKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 전문 검색하고 원문별 최고 점수 청크를 반환합니다."""
    # state에 질문, questions에 언어 선택 기준을 전달합니다.
    decision = language_classifier.invoke({
        "state": query, "questions": {"language": language_question},
    })
    language = decision.choices["language"].choice
    # 한글은 영어 검색어 목록을 받아 OR로 연결하고, 영어는 그대로 사용합니다.
    if language == "ko":
        terms = lexical_chain.invoke({"question": query})
        search_text = " OR ".join(f'"{word}"' for word in terms.keywords)
    else:
        search_text = query
    print("질문 언어:", language, "| 전문 검색어:", search_text)
    # 전문 인덱스에서 검색어와 일치하는 청크를 최대 top_k개 조회합니다.
    hits = run_cypher("""
        MATCH (node:Chunk)
        // 전문 인덱스에서 검색어와 관련 있는 청크를 최대 top_k개 찾습니다.
        SEARCH node IN (
            FULLTEXT INDEX movie_fulltext
            FOR $search_text
            LIMIT $top_k
        ) SCORE AS score
        // 결과 통합에 쓸 원문 ID를 반환합니다.
        RETURN node.source_id AS source_id, node.id AS chunk_id, score
        // 점수 내림차순으로 정렬합니다.
        ORDER BY score DESC, chunk_id
    """, search_text=search_text, top_k=top_k)
    rows, seen = [], set()
    # 영화별 순위를 비교하도록 원문마다 최고 점수 검색 행을 대표로 남깁니다.
    for row in hits:
        source_id = row["source_id"]
        if source_id not in seen:
            seen.add(source_id)
            rows.append(row)
    return rows


### 점수 대신 순위로 기여를 계산합니다

가중 RRF는 검색 순위 `r`을 `가중치 / (60 + r)`로 바꾸고, 같은 문서의 점수를 합칩니다.
이때 한 검색 방식이 더하는 점수를 **기여 점수**라고 부릅니다.
전문·벡터 검색은 문서 순위를 사용합니다. 관계 검색은 배우·연도 조건을 만족하는 목록이므로 관련성 순위가 없습니다.
따라서 이번 실습에서는 **관계 검색으로 찾은 모든 문서에 `r=1`을 적용해 같은 점수를 더합니다.**

같은 검색 방식에서 중복된 문서는 한 번만 계산합니다. 여러 방식이 찾은 문서는 각 방식의 점수를 더합니다.
관계 검색은 전문·벡터 검색이 놓친 영화를 후보에 추가할 수 있지만, 관계 점수만으로 후보 사이의 순위가 갈리지는 않습니다.

전문·벡터 검색 순위로 점수를 계산하고 관계 후보에는 같은 점수를 더하는 RRF 함수를 만듭니다.


In [ ]:
def fuse_routes(routes, weights, c=60):
    """검색 방식별 RRF 점수를 문서별로 합칩니다. 관계 검색으로 찾은 문서에는 같은 점수를 더합니다.

    Args:
        routes: 전문(fulltext)·벡터(dense)·관계(graph) 검색 결과를 담은 사전.
        weights: 각 검색 방식의 반영 비율. 이번 실습에서는 합을 1로 둡니다.
        c: 순위 간 점수 차이를 줄이는 상수. 클수록 순위의 영향이 줄어듭니다.
    Returns:
        원문 ID, RRF 총점, 각 검색 방식이 더한 점수를 담은 문서 목록.
    """
    merged = {}
    for route, rows in routes.items():
        # 검색 방식이 바뀔 때마다 중복 문서 기록을 새로 시작합니다.
        seen = set()
        unique_rank = 0
        for row in rows:
            source_id = row["source_id"]
            # 이 검색 방식에서 이미 계산한 원문 ID는 건너뜁니다.
            if source_id in seen:
                continue
            seen.add(source_id)
            # 청크 중복을 제거한 문서 순위를 1부터 셉니다.
            unique_rank += 1
            # 관계 검색에는 순위가 없으므로 모든 문서의 rank를 1로 둡니다.
            rank = 1 if route == "graph" else unique_rank
            # 이 검색 방식이 해당 문서에 더할 RRF 점수를 계산합니다.
            contribution = weights[route] / (c + rank)
            # 처음 나온 문서는 점수 0으로 만들고, 이미 나온 문서는 누적 점수를 유지합니다.
            target = merged.setdefault(source_id, {
                "source_id": source_id, "rrf_score": 0.0, "fulltext": 0.0, "dense": 0.0, "graph": 0.0
            })
            # 현재 검색 방식의 점수를 기록하고 문서의 총점에 더합니다.
            target[route] = contribution
            target["rrf_score"] += contribution
    # RRF 총점 내림차순으로 반환합니다.
    return sorted(merged.values(), key=lambda row: (-row["rrf_score"], row["source_id"]))


전문·벡터 검색 결과와 관계 후보를 모아 가중 RRF로 통합합니다.


In [ ]:
# 한글 질문을 영어 검색어로 바꿔 전문 검색합니다.
fulltext_rows = fulltext_documents(semantic_query, top_k=30)
dense_rows = vector_documents(query_vector, top_k=30)
routes = {"fulltext": fulltext_rows, "dense": dense_rows, "graph": graph_rows}
weights = {"fulltext": 0.35, "dense": 0.45, "graph": 0.20}
fused_rows = fuse_routes(routes, weights)
display(pd.DataFrame(fused_rows))


### 후보 예산과 최종 문서 수를 구분합니다

여기서 **검색 예산**은 각 검색 방식에서 가져올 결과 수의 상한입니다.
`top_k=30`으로 최대 청크 30개를 가져와도 같은 원문을 합치고 배우·연도 조건을 적용하면 영화 수는 줄어듭니다.
따라서 검색할 청크 수와 답변에 사용할 최종 영화 3편을 구분합니다.
원하는 영화가 없으면 각 검색 결과에 포함됐는지, 가져올 청크 수가 충분한지 먼저 확인하세요.
데이터가 커지면 지원되는 검색 전 필터나 관계 조건을 먼저 적용하는 설계도 비교합니다.
이번 38편 데이터의 관계 경로는 조건에 맞는 문서를 모두 가져와 어휘·의미 후보 누락을 보완합니다.

### 관련성 점수와 필수 조건을 구분합니다

관계 경로의 가중치를 올리는 것만으로 필수 조건을 보장할 수는 없습니다.
높은 검색 점수로 들어온 조건 밖 영화가 남을 수 있으므로, 명시한 관계·연도 조건을 마지막에 적용합니다.
관계 검색 점수는 순위를 정하는 데 쓰고, `allowed_ids`는 조건에 맞는 영화를 남기는 데 씁니다.

배우·연도 조건의 통과·제외 수와 남은 작품을 확인합니다.


In [ ]:
# 배우·연도 조건을 만족하는 후보만 남깁니다.
candidate_rows = [row for row in fused_rows if row["source_id"] in allowed_ids]
removed_rows = [row for row in fused_rows if row["source_id"] not in allowed_ids]
print("통합 문서:", len(fused_rows), "조건 통과:", len(candidate_rows), "조건 제외:", len(removed_rows))
show_documents(candidate_rows)


### 단독 검색과 통합 검색의 차이를 확인합니다

검색은 다시 호출하지 않습니다. 이미 받은 `routes`에서 사용할 경로만 바꿉니다.
모든 조합에 같은 배우·연도 조건을 적용하고, PageRank를 넣기 전 상위 3편을 비교합니다.
선택한 경로의 가중치 합을 1로 맞추되, **서로 다른 조합의 RRF 점수 크기로 우열을 판단하지 않습니다.**

관계 경로가 새로 포함한 영화가 있는지, 원문을 읽었을 때 답에 필요한 영화인지 확인하세요.
후보 수가 늘거나 정답 후보 순위가 같을 수도 있습니다. 통합이 항상 개선된다고 가정하지 않습니다.

전문·벡터·관계 검색을 합치는 조합별로 후보 수와 상위 작품을 비교합니다.


In [ ]:
# 전문·벡터·관계 검색을 어떤 조합으로 합칠지 정합니다.
route_sets = {
    "전문 검색": ["fulltext"], "벡터": ["dense"],
    "전문+벡터": ["fulltext", "dense"], "전문+벡터+관계": ["fulltext", "dense", "graph"],
}
# 이미 받은 검색 결과를 골라 조합별 후보 수와 순위를 비교합니다.
route_comparison = []
for label, names in route_sets.items():
    selected = {name: routes[name] for name in names}
    # 이번 조합에 포함된 검색 방식의 가중치 합을 1로 맞춥니다.
    total_weight = sum(weights[name] for name in names)
    selected_weights = {name: weights[name] / total_weight for name in names}
    fused = fuse_routes(selected, selected_weights)
    # 같은 배우·연도 조건을 적용합니다.
    passed = [row for row in fused if row["source_id"] in allowed_ids]
    route_comparison.append({
        "경로 조합": label, "고유 후보": len(fused), "조건 통과": len(passed),
        "상위 원문 ID": [row["source_id"] for row in passed[:3]],
        "상위 제목": [documents_by_id[row["source_id"]]["title"] for row in passed[:3]],
    })
display(pd.DataFrame(route_comparison))


### 🖐️ 함께 따라하기: 섬 생존 줄거리와 관계 조건을 합칩니다

`practice_semantic`을 `비행기 추락 사고에서 살아남은 남자가 외딴 무인도에서 홀로 생존하는 영화`로 두세요. 전문 검색·Dense는 청크 30개, 가중치는 시연과 같게 합니다. 세 검색 결과를 `practice_routes`에 담으세요. 관계 경로는 `practice_graph_rows`, 필수 조건은 `practice_eligible`의 ID를 사용합니다. 최종 통과 후보를 `practice_candidates`에 담으세요.

무인도 생존 질문의 세 검색 결과를 합치고 배우·연도 조건을 만족하는 영화만 남깁니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### 🖐️ 함께 따라하기: 관계 경로가 보완한 문서를 찾습니다

이미 조회한 `practice_routes`를 사용합니다. 전문 검색과 벡터 검색의 합집합 ID를 `practice_text_ids`, 관계 검색 ID를 `practice_graph_ids`로 만드세요. 관계 검색에만 있는 문서와 그 원문을 확인하세요. 차집합이 비어 있으면 그 사실을 적고, 후보 수 증가 없이도 관계 조건 확인이 필요한 이유를 설명하세요. 새 임베딩이나 검색 호출은 하지 않습니다.

전문·벡터 검색이 놓치고 관계 검색만 찾은 원문을 확인합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### ✅ 바로 확인 퀴즈

배우·연도 조건을 지키려면 그래프 가중치를 아주 높이면 충분할까요?

<details><summary>해설 보기</summary>

가중치는 상대 점수이므로 필수 조건을 보장하지 않습니다. 조건을 만족하는 ID 집합으로 후보를 제한해야 합니다.

</details>


## 3. 같은 통합 후보 안에서 PageRank의 영향을 비교합니다

<img src="images/03_pagerank_multihop.png" width="1100" alt="출연 그래프의 PageRank를 조건을 통과한 통합 후보에만 적용하고, 답변에는 원문과 출연 관계를 별도 근거로 전달합니다.">

PageRank는 출연 관계로 연결된 영화·인물의 그래프 내 중요도를 계산합니다.
GDS가 계산할 수 있도록 영화·인물과 `ACTED_IN`을 메모리 그래프로 옮기는 과정을 **투영**이라고 합니다.
계산할 때만 출연 관계를 양방향으로 다루며, DB에 저장된 방향은 유지합니다.
이번에는 **조건을 통과한 같은 후보**에 영화 자체의 PageRank를 적용하고 순위 변화를 비교합니다.

[GDS PageRank 공식 문서](https://neo4j.com/docs/graph-data-science/current/algorithms/page-rank/)

출연 그래프에서 영화의 PageRank 점수를 계산하고 원문 ID에 연결합니다.


In [ ]:
# PageRank 계산에 사용할 메모리 그래프의 이름을 정합니다.
projection_name = "movie_cast"
# 같은 이름의 메모리 그래프를 지웁니다. DB의 노드·관계는 유지됩니다.
run_cypher("CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName", name=projection_name)
# 영화·인물의 출연 관계로 PageRank 계산용 메모리 그래프를 만듭니다.
run_cypher("""
    CALL gds.graph.project($name, ['Movie', 'Person'],
        // 계산할 때 출연 관계를 양방향으로 다룹니다.
        {ACTED_IN: {orientation: 'UNDIRECTED'}})
    YIELD nodeCount, relationshipCount RETURN nodeCount, relationshipCount
""", name=projection_name)
# PageRank를 계산해 pagerank 속성에 저장합니다.
run_cypher("""
    CALL gds.pageRank.write($name, {writeProperty: 'pagerank', dampingFactor: 0.85})
    YIELD nodePropertiesWritten RETURN nodePropertiesWritten
""", name=projection_name)

# 각 원문이 설명하는 영화의 PageRank 점수를 조회합니다.
pagerank_rows = run_cypher("""
    MATCH (d:Document)-[:ABOUT_MOVIE]->(m:Movie)
    RETURN d.id AS source_id, m.title AS title, m.pagerank AS pagerank
    ORDER BY pagerank DESC
""")
# source_id로 해당 영화의 PageRank 점수를 찾을 수 있게 만듭니다.
pagerank_by_id = {row["source_id"]: row["pagerank"] for row in pagerank_rows}
display(pd.DataFrame(pagerank_rows).head())


검색 점수와 PageRank를 0~1로 바꾸고 비중에 따라 합쳐 영화 순위를 정하는 함수를 만듭니다.


In [ ]:
def minmax(values):
    """현재 후보 안에서 값을 0~1로 변환합니다. 동점만 있으면 모두 0.5입니다."""
    low, high = min(values), max(values)
    # 모두 동점이면 0.5로 통일합니다.
    return [0.5 if high == low else (value - low) / (high - low) for value in values]


def rerank_candidates(rows, weight):
    """같은 후보에 RRF와 영화 PageRank를 결합합니다.

    Args:
        rows: 관계 조건을 통과한 통합 후보.
        weight: PageRank 비중. 0이면 RRF만 반영합니다.
    Returns:
        0~1로 바꾼 두 점수와 최종 점수(final_score)를 붙여 정렬한 목록.
    """
    # 조건을 통과한 후보가 없으면 재정렬 없이 빈 목록을 반환합니다.
    if not rows:
        return []
    # 같은 후보에서 RRF와 PageRank 점수를 각각 0~1로 바꿉니다.
    retrieval = minmax([row["rrf_score"] for row in rows])
    centrality = minmax([pagerank_by_id[row["source_id"]] for row in rows])
    ranked = []
    # 각 후보에 그 문서의 검색 점수와 PageRank 점수를 연결합니다.
    for row, relevance, importance in zip(rows, retrieval, centrality):
        # weight 비율만큼 PageRank를 반영해 새 점수를 만듭니다.
        ranked.append({**row, "rrf_norm": relevance, "pagerank_norm": importance,
                       "final_score": (1 - weight) * relevance + weight * importance})
    # 같은 후보를 결합 점수순으로 재정렬합니다.
    return sorted(ranked, key=lambda row: (-row["final_score"], row["source_id"]))


PageRank 비중을 0·0.2·0.6으로 바꾸며 같은 후보의 순위를 비교합니다.


In [ ]:
# 후보 집합과 검색 점수는 고정하고 PageRank 비중만 비교합니다.
comparison = []
for weight in [0.0, 0.2, 0.6]:
    ranked = rerank_candidates(candidate_rows, weight)
    for rank, row in enumerate(ranked[:3], start=1):
        comparison.append({"pagerank_weight": weight, "rank": rank,
                           "title": documents_by_id[row["source_id"]]["title"], **row})
display(pd.DataFrame(comparison))


**비교의 질문:** `Johnny Mnemonic`의 순위가 유지되나요? 중심성을 늘리면서
뇌에 데이터를 운반하는 줄거리와 무관한 영화가 올라오나요? 실제 출력과 원문으로 판단하세요.
순위가 그대로인 것도 관찰 결과입니다. PageRank 비중 0.2는 비교할 설정이며 정답값이 아닙니다.


### 🖐️ 함께 따라하기: 공유 배우가 많아도 줄거리는 다를 수 있습니다

`practice_candidates`를 그대로 고정해 PageRank 비중 0·0.6의 전체 후보 순위를 비교하세요. `practice_eligible`의 공통 배우도 함께 읽습니다. `Cast Away`와 `The Green Mile` 중 섬 생존 조건을 뒷받침하는 원문이 어느 쪽인지 설명하고, 유지할 비중을 적으세요.

PageRank 비중 0·0.6의 순위를 줄거리·공유 배우 근거와 대조합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### ✅ 바로 확인 퀴즈

모든 질문에서 같은 영화가 계속 올라온다면 어느 단계를 확인해야 할까요?

<details><summary>해설 보기</summary>

질문과 무관한 전역 중심성의 비중, 후보 정규화 범위, 필수 조건을 확인합니다. 중심성이 관련성을 덮고 있다면 비중을 줄이거나 0으로 두고 원문을 다시 비교합니다.

</details>


## 4. 멀티홉 경로와 원문을 구분해 답변합니다

관계 경로는 `The Matrix <- Keanu Reeves -> Johnny Mnemonic`처럼 **왜 후보가 조건에 맞는지** 설명합니다.
뇌에 기밀 데이터를 운반한다는 줄거리는 Wikipedia 원문에서 확인합니다.
영화의 개봉연도는 Movies 레코드 속성에서 확인합니다.

각 근거에 `evidence_id`를 붙여 답변의 인용과 원문을 연결합니다.
상위 3편의 줄거리 원문·출연 관계·개봉연도를 모으고, 같은 출연 관계는 한 번만 넣습니다.

`build_evidence`는 상위 영화의 줄거리·연도를 담은 뒤, 중복을 뺀 출연 관계 ID로 근거 문장을 가져옵니다.
제목·연도·출연 경로는 앞서 조회한 `eligible_rows`를 사용합니다. 줄거리·출연 근거 문장은 DB에 적재한 동일한 배포 패킷에서 읽습니다.

| 근거 종류 | evidence_id | 확인할 내용 |
|---|---|---|
| `plot` | 기존 원문 `doc_id` | 후보 영화의 줄거리 |
| `cast_relation` | 기존 관계 `claim_id` | 배우가 기준 영화와 후보 영화에 각각 출연했는지 |
| `movie_record` | `movie:` + 기존 영화 `standard_id` | 영화 제목과 개봉연도 |

`movie:`는 답변 근거의 종류를 구별하기 위한 접두어입니다. DB의 기존 영화 ID는 바꾸지 않습니다.

줄거리·출연·연도 근거를 모으는 함수와 근거를 인용하는 답변 체인을 준비합니다.


In [ ]:
# claim_id로 출연 근거를 찾을 사전을 만듭니다.
relations_by_id = {row["claim_id"]: row for row in movies["relations"]}


def build_evidence(ranked, eligible_rows, top_n=3):
    """상위 영화의 줄거리·개봉연도·출연 근거를 모읍니다."""
    # 후보 원문 ID로 영화 속성과 출연 경로를 찾습니다.
    eligible_by_id = {row["source_id"]: row for row in eligible_rows}
    evidence = []
    claim_ids = set()

    # 상위 영화의 줄거리와 이미 조회한 제목·개봉연도를 담습니다.
    for row in ranked[:top_n]:
        source_id = row["source_id"]
        doc = documents_by_id[source_id]
        movie = eligible_by_id[source_id]
        evidence.extend([
            {"evidence_id": source_id, "kind": "plot", "title": doc["title"],
             "url": doc["url"], "text": doc["text"]},
            {"evidence_id": "movie:" + movie["movie_id"], "kind": "movie_record",
             "url": "https://github.com/neo4j-graph-examples/movies",
             "text": f"{movie['title']} / 개봉연도: {movie['released']}"},
        ])
        # 양쪽 출연 관계 ID를 집합에 모으면 중복이 자동으로 제거됩니다.
        for path in movie["paths"]:
            claim_ids.update([path["first_claim"], path["second_claim"]])

    # 모아 둔 관계 ID로 실제 근거 문장과 출처를 가져옵니다.
    for claim_id in sorted(claim_ids):
        claim = relations_by_id[claim_id]
        evidence.append({"evidence_id": claim_id, "kind": "cast_relation",
                         "url": claim["source_url"], "text": claim["evidence"]})
    return evidence


# 줄거리·연도·출연 근거를 인용해 답하도록 프롬프트를 만듭니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공한 영화 데이터만 사용해 한국어로 답하세요. 줄거리와 배우·연도 조건을 모두 확인하세요. "
               "답변은 세 항목으로 작성하세요. 1. 작품과 개봉연도, 2. 기준 영화와 공유하는 배우, 3. 질문에 맞는 줄거리. "
               "각 항목의 문장 바로 뒤에 실제 evidence_id를 인용하세요. 대괄호 하나에는 ID 하나만 넣으세요. 여러 ID는 [ID1][ID2] 형태로 붙이고 쉼표로 묶지 마세요. "
               "연도에는 movie_record, 줄거리에는 plot, 공유 배우마다 양쪽 cast_relation의 ID를 인용하세요. "
               "예: - 작품과 연도: 작품명과 연도 [해당 영화 속성 ID]. "
               "evidence_ids 목록에만 ID를 적는 것은 충분하지 않습니다. answer 본문에도 반드시 표시하세요. "
               "배우 본인이 사고를 겪었다고 쓰지 말고 배우와 극중 인물을 구분하세요. "
               "조건은 맞지만 줄거리가 다른 후보를 선택하지 마세요. 부족한 근거는 부족하다고 말하세요. "
               "evidence_ids에는 answer 본문에 인용한 모든 ID를 빠짐없이 한 번씩 담으세요. ID마다 대괄호를 따로 쓰세요. 질문의 줄거리 조건에 맞는 영화가 없으면 유사한 다른 영화를 정답으로 제시하지 마세요. 각 항목은 한 문장으로 쓰고, 줄거리는 질문의 조건에 해당하는 핵심 사건만 요약하세요."),
    ("human", "질문: {question}\n근거: {context}"),
])
# 답변 본문과 인용 ID 목록을 구조화합니다.
class GroundedAnswer(BaseModel):
    answer: str = Field(description="작품·연도, 공유 배우, 줄거리의 세 항목. 각 항목은 한 문장. 본문에 [실제 evidence_id]를 표시하며 대괄호 하나에 ID 하나만 넣은 한국어 답변")
    evidence_ids: list[str] = Field(description="answer 본문의 모든 대괄호 ID를 빠짐없이 담은 목록. 줄거리의 plot 원문 ID, 영화 속성의 movie_record ID, 공유 배우 양쪽 cast_relation ID를 모두 포함합니다.")


# 답변 체인을 구성합니다. invoke에서 실제 호출합니다.
answer_chain = answer_prompt | llm.with_structured_output(GroundedAnswer)


상위 후보의 줄거리·출연·연도 근거를 모아 종류와 출처를 확인합니다.


In [ ]:
# PageRank 비중을 0.2로 정해 답변에 사용할 후보 순서를 만듭니다.
final_ranked = rerank_candidates(candidate_rows, weight=0.2)
evidence = build_evidence(final_ranked, eligible_rows, top_n=3)
display(pd.DataFrame(evidence)[["evidence_id", "kind", "url"]])
print("전달 근거 글자 수:", sum(len(row["text"]) for row in evidence))


질문과 근거를 모델에 전달해 답변과 인용 ID를 생성합니다.


In [ ]:
# 질문과 근거를 전달해 답변을 생성합니다.
answer = answer_chain.invoke({"question": question, "context": json.dumps(evidence, ensure_ascii=False)})
print(answer.answer)
print("인용 ID:", answer.evidence_ids)


인용 ID의 존재 여부와 본문 표시를 확인하고 인용 원문을 읽습니다.


In [ ]:
# 답변이 인용한 ID로 실제 근거를 찾을 사전을 만듭니다.
evidence_by_id = {row["evidence_id"]: row for row in evidence}
# 본문과 모델 목록 양쪽의 인용 ID를 확인합니다.
inline_ids = set(re.findall(r"\[([^\[\]]+)\]", answer.answer))
unknown_ids = (inline_ids | set(answer.evidence_ids)) - set(evidence_by_id)
# ^는 본문과 인용 목록 중 한쪽에만 있는 근거 ID를 고릅니다.
citation_gap = inline_ids ^ set(answer.evidence_ids)
print("본문과 목록에 인용 있음:", bool(inline_ids) and bool(answer.evidence_ids))
print("제공하지 않은 인용 ID:", unknown_ids)
print("본문과 목록의 차이:", citation_gap)
for evidence_id in answer.evidence_ids:
    if evidence_id in evidence_by_id:
        item = evidence_by_id[evidence_id]
        print("\n근거 ID:", evidence_id, "| 종류:", item["kind"], "| 출처:", item["url"])
        print(item["text"])


### 🖐️ 함께 따라하기: 섬 생존 질문의 답변과 인용을 대조합니다

`practice_candidates`를 비중 0으로 재정렬하고 상위 3편의 근거를 `practice_evidence`에 모으세요. Apollo 13·공유 배우·1999~2000·섬 생존을 모두 포함한 한국어 질문으로 답변을 한 번 만드세요. Cast Away의 줄거리, Tom Hanks의 양쪽 출연 관계, 개봉연도 근거를 각각 찾아 읽습니다.

무인도 생존 질문에 답변을 생성하고 실제 인용 근거와 대조합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


## 5. 새 조건에서 누락이 시작된 단계를 찾습니다

다음 질문은 공유 배우 수 조건까지 바뀝니다.

> Sleepless in Seattle과 출연 배우를 최소 2명 공유하고 1993년 이후 개봉한 영화 중,
> 서로 경쟁하는 서점 주인들이 익명 이메일로 사랑하게 되는 작품을 찾아 주세요.

`You've Got Mail`의 원문과 `Tom Hanks`·`Meg Ryan`의 출연 관계가 판단 근거입니다.
`Joe Versus the Volcano`는 공유 배우만 보고 포함하면 안 됩니다. 개봉연도 조건도 확인하세요.
이 정보는 순위 결과가 아니라 **기존 데이터에서 확인할 기준**입니다.


### 🖐️ 함께 따라하기: 같은 통합 검색을 새 질문에 적용합니다

줄거리 검색어는 `경쟁하는 서점 주인들이 서로의 정체를 모른 채 익명 이메일로 사랑하게 되는 영화`를 사용합니다. 기준 영화 `Sleepless in Seattle`, 최소 연도 1994, 최대 연도 9999, 최소 공유 배우 2명을 설정하세요. 관계 체인·전문 검색·Dense·융합·필수 조건·PageRank 비중 0·근거 조립까지 연결하고, 각 단계의 고유 문서 수를 출력하고 `new_evidence`에 근거를 모으세요. 이 셀에서는 원문까지 확인합니다.

새 질문으로 검색한 결과를 합치고 배우 수·연도 조건에 맞는 상위 영화의 근거를 모읍니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### 원문을 읽은 뒤 답변을 요청합니다

위 결과의 줄거리·공통 배우·개봉연도를 확인한 뒤 실행합니다.
`new_question`에는 위의 한국어 질문 전체를 넣고, `new_evidence`를 전달해 `new_answer`를 만드세요.
모델을 한 번 호출하고 답변과 실제 인용 ID를 읽습니다.

익명 이메일 영화 질문의 답변을 생성하고 인용 ID와 근거를 대조합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### 결과가 부족하면 어느 단계를 고칠까요?

| 관찰 결과 | 먼저 확인할 곳 |
|---|---|
| 필요한 영화가 모든 후보에 없음 | 청크 예산·검색 언어·생성 Cypher의 조건 |
| 통합 후보에는 있지만 조건 적용 뒤 사라짐 | 기준 영화·연도 경계·공유 배우 수 |
| 후보에는 있지만 최종 상위에서 밀림 | 경로 가중치·PageRank 비중·최종 문서 수 |
| 영화는 맞지만 배우나 연도 설명이 틀림 | 관계 ID·영화 속성·LLM 인용 |
| 인용 ID는 있으나 주장과 관계없음 | 인용 원문이 실제 주장을 뒷받침하는지 |

한 번에 하나의 설정만 바꾸고 같은 질문·후보·최종 개수를 유지해 비교하세요.


### ✅ 바로 확인 퀴즈

후보에 없는 영화를 PageRank 재정렬이나 답변 프롬프트 수정으로 되찾을 수 있나요?

<details><summary>해설 보기</summary>

재정렬은 현재 후보의 순서만 바꿉니다. 후보 검색·관계 조건·확장 한도에서 누락이 시작된 곳을 먼저 고쳐야 합니다.

</details>


## 이번 강의 정리

- 세 경로의 반환 단위를 원문 ID로 통일하고, 경로 내부 중복을 제거했습니다.
- 순위가 없는 관계 조건 집합과 관련성 순위를 구분했습니다.
- 조건 통과 후보를 고정한 뒤 PageRank의 영향을 비교했습니다.
- 출연 경로·줄거리 원문·개봉연도 근거를 나누어 답변과 인용을 대조했습니다.


## 교안 02 핵심 코드 이어서 보기

본문에서 그래프와 인덱스를 적재한 뒤 실행합니다.


DB·모델을 연결하고 답변에 사용할 원문·관계 근거를 불러옵니다.


In [ ]:
import json
import os
import re
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.types import RetrieverResultItem
from pydantic import BaseModel, Field

# 교안의 .env로 DB와 질문 임베딩 모델을 연결합니다.
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 질문을 자동으로 나누지 않습니다.
)

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


# 답변에 인용할 원문과 관계 근거를 불러옵니다.
movies = json.loads((material_dir / "data/movies_extraction_packet.json").read_text(encoding="utf-8"))
documents_by_id = movies["documents"]


검색된 청크에서 영화별 대표 후보를 고르는 함수를 만듭니다.


In [ ]:
def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # content는 청크 본문, metadata는 청크 ID·원문 ID·유사도를 담는 사전입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
vector_retriever = VectorRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    result_formatter=format_vector,
)


def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 영화별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    result = vector_retriever.search(query_vector=query_vector, top_k=top_k)
    # 영화별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(dict(row))
    return rows


질문의 언어에 맞춰 전문 검색을 수행하는 함수를 만듭니다.


In [ ]:
# Jev가 한글·영어 중 질문의 언어를 선택하도록 설정합니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영어 영화명 등이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


# 영어 검색어를 문자열 목록으로 받습니다.
class EnglishKeywords(BaseModel):
    """영어 원문 검색에 사용할 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 영어 단어 4~8개")


# 한글 질문에서 영어 검색어 목록을 만드는 체인입니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "한글 영화 줄거리 질문을 영어 원문 검색용 핵심어 4~8개로 바꾸세요. "
               "질문의 핵심 사건·대상을 나타내는 단어를 골라 keywords의 각 항목에 영어 단어 하나씩 담으세요. "
               "영화 정답이나 배우를 추측해 추가하지 마세요. 질문에 없는 고유명사는 넣지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(EnglishKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 전문 검색하고 원문별 최고 점수 청크를 반환합니다."""
    # state에 질문, questions에 언어 선택 기준을 전달합니다.
    decision = language_classifier.invoke({
        "state": query, "questions": {"language": language_question},
    })
    language = decision.choices["language"].choice
    # 한글은 영어 검색어 목록을 받아 OR로 연결하고, 영어는 그대로 사용합니다.
    if language == "ko":
        terms = lexical_chain.invoke({"question": query})
        search_text = " OR ".join(f'"{word}"' for word in terms.keywords)
    else:
        search_text = query
    # 전문 인덱스에서 검색어와 일치하는 청크를 최대 top_k개 조회합니다.
    hits = run_cypher("""
        MATCH (node:Chunk)
        // 전문 인덱스에서 검색어와 관련 있는 청크를 최대 top_k개 찾습니다.
        SEARCH node IN (
            FULLTEXT INDEX movie_fulltext
            FOR $search_text
            LIMIT $top_k
        ) SCORE AS score
        // 결과 통합에 쓸 원문 ID를 반환합니다.
        RETURN node.source_id AS source_id, node.id AS chunk_id, score
        // 점수 내림차순으로 정렬합니다.
        ORDER BY score DESC, chunk_id
    """, search_text=search_text, top_k=top_k)
    rows, seen = [], set()
    # 영화별 순위를 비교하도록 원문마다 최고 점수 검색 행을 대표로 남깁니다.
    for row in hits:
        source_id = row["source_id"]
        if source_id not in seen:
            seen.add(source_id)
            rows.append(row)
    return rows


관계 질문을 Cypher로 바꿔 조회하는 체인을 만듭니다.


In [ ]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

# 앞에서 설정한 실습 DB 계정으로 연결합니다.
graph = Neo4jGraph(
    url=neo4j_uri,
    username=os.environ["NEO4J_USER"],
    password=os.environ["NEO4J_PASSWORD"],
    enhanced_schema=False,  # 속성 예시 값 수집을 생략하고 기본 스키마를 사용합니다.
)
# 관계 질문을 지정한 스키마의 조회 Cypher로 바꾸는 규칙을 정합니다.
cypher_prompt = PromptTemplate.from_template("""
다음 스키마만 사용해 조회용 Cypher 하나를 작성하세요.
{schema}
Movie, Person, Document를 사용하세요. ACTED_IN은 인물에서 영화 방향입니다.
Document는 ABOUT_MOVIE로 영화에 연결됩니다. Document.id가 source_id입니다.
영화명이 한글과 영문으로 함께 제시되면 DB의 영문 title을 사용하세요.
공유 배우 수는 count(DISTINCT actor)로 세고 기준 영화 자신은 제외하세요.
기준 영화 제외·개봉연도 조건은 집계 WITH 앞의 WHERE에서 적용하세요.
WITH 뒤에서 사용할 변수는 WITH에 포함하고, 집계한 공유 배우 수 조건은 WITH 뒤에서 적용하세요.
RETURN DISTINCT d.id AS source_id, d.title AS title을 사용하세요.
ORDER BY source_id LIMIT 100으로 일정한 순서로 출력하세요.
줄거리 유사도를 그래프 속성으로 가정하지 마세요. Cypher만 반환하세요.
질문: {question}
""")
# 질문 → Cypher 생성 → DB 조회를 연결합니다.
graph_chain = GraphCypherQAChain.from_llm(
    llm=llm, graph=graph, cypher_prompt=cypher_prompt,
    # 필요한 레이블·관계 유형만 프롬프트에 넣습니다.
    include_types=["Movie", "Person", "Document", "ACTED_IN", "ABOUT_MOVIE"],
    # 조회 결과를 최대 100행 반환합니다.
    return_direct=True, return_intermediate_steps=False, top_k=100,
    # 쿼리 실행에 동의합니다. 쓰기 권한을 제한하지는 않습니다.
    allow_dangerous_requests=True,
)


배우·연도 조건과 양쪽 출연 근거를 조회할 Cypher를 작성합니다.


In [ ]:
# 배우·연도 조건과 양쪽 출연 근거를 조회할 쿼리입니다.
eligible_query = """
// 같은 배우를 공유하는 영화 쌍을 찾습니다.
MATCH (anchor:Movie {title: $anchor})<-[r1:ACTED_IN]-(actor:Person)-[r2:ACTED_IN]->(candidate:Movie)
// 연도는 양쪽 경계를 포함하며, 기준 영화 자신은 후보에서 제외합니다.
WHERE candidate <> anchor AND candidate.released >= $min_year AND candidate.released <= $max_year
// 고유 배우 수를 세고 양쪽 출연 관계 ID를 모읍니다.
WITH candidate, count(DISTINCT actor) AS shared_count,
     collect(DISTINCT {actor: actor.name, first_claim: r1.claim_id, second_claim: r2.claim_id}) AS paths
// 집계한 공유 배우 수로 후보를 제한합니다.
WHERE shared_count >= $min_shared
// 영화 후보를 검색 결과와 같은 원문 ID로 연결합니다.
MATCH (d:Document)-[:ABOUT_MOVIE]->(candidate)
RETURN d.id AS source_id, candidate.title AS title, candidate.released AS released,
       candidate.standard_id AS movie_id, shared_count, paths
// 관련성 순위 없이 ID순으로 출력합니다.
ORDER BY source_id
"""


세 검색 결과를 문서별 RRF 점수로 합치는 함수를 만듭니다.


In [ ]:
def fuse_routes(routes, weights, c=60):
    """검색 방식별 RRF 점수를 문서별로 합칩니다. 관계 검색으로 찾은 문서에는 같은 점수를 더합니다.

    Args:
        routes: 전문(fulltext)·벡터(dense)·관계(graph) 검색 결과를 담은 사전.
        weights: 각 검색 방식의 반영 비율. 이번 실습에서는 합을 1로 둡니다.
        c: 순위 간 점수 차이를 줄이는 상수. 클수록 순위의 영향이 줄어듭니다.
    Returns:
        원문 ID, RRF 총점, 각 검색 방식이 더한 점수를 담은 문서 목록.
    """
    merged = {}
    for route, rows in routes.items():
        # 검색 방식이 바뀔 때마다 중복 문서 기록을 새로 시작합니다.
        seen = set()
        unique_rank = 0
        for row in rows:
            source_id = row["source_id"]
            # 이 검색 방식에서 이미 계산한 원문 ID는 건너뜁니다.
            if source_id in seen:
                continue
            seen.add(source_id)
            # 청크 중복을 제거한 문서 순위를 1부터 셉니다.
            unique_rank += 1
            # 관계 검색에는 순위가 없으므로 모든 문서의 rank를 1로 둡니다.
            rank = 1 if route == "graph" else unique_rank
            # 이 검색 방식이 해당 문서에 더할 RRF 점수를 계산합니다.
            contribution = weights[route] / (c + rank)
            # 처음 나온 문서는 점수 0으로 만들고, 이미 나온 문서는 누적 점수를 유지합니다.
            target = merged.setdefault(source_id, {
                "source_id": source_id, "rrf_score": 0.0, "fulltext": 0.0, "dense": 0.0, "graph": 0.0
            })
            # 현재 검색 방식의 점수를 기록하고 문서의 총점에 더합니다.
            target[route] = contribution
            target["rrf_score"] += contribution
    # RRF 총점 내림차순으로 반환합니다.
    return sorted(merged.values(), key=lambda row: (-row["rrf_score"], row["source_id"]))


영화의 PageRank 점수를 계산합니다.


In [ ]:
# PageRank 계산에 사용할 메모리 그래프의 이름을 정합니다.
projection_name = "movie_cast"
# 같은 이름의 메모리 그래프를 지웁니다. DB의 노드·관계는 유지됩니다.
run_cypher("CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName", name=projection_name)
# 영화·인물의 출연 관계로 PageRank 계산용 메모리 그래프를 만듭니다.
run_cypher("""
    CALL gds.graph.project($name, ['Movie', 'Person'],
        // 계산할 때 출연 관계를 양방향으로 다룹니다.
        {ACTED_IN: {orientation: 'UNDIRECTED'}})
    YIELD nodeCount, relationshipCount RETURN nodeCount, relationshipCount
""", name=projection_name)
# PageRank를 계산해 pagerank 속성에 저장합니다.
run_cypher("""
    CALL gds.pageRank.write($name, {writeProperty: 'pagerank', dampingFactor: 0.85})
    YIELD nodePropertiesWritten RETURN nodePropertiesWritten
""", name=projection_name)

# 각 원문이 설명하는 영화의 PageRank 점수를 조회합니다.
pagerank_rows = run_cypher("""
    MATCH (d:Document)-[:ABOUT_MOVIE]->(m:Movie)
    RETURN d.id AS source_id, m.title AS title, m.pagerank AS pagerank
    ORDER BY pagerank DESC
""")
# source_id로 해당 영화의 PageRank 점수를 찾을 수 있게 만듭니다.
pagerank_by_id = {row["source_id"]: row["pagerank"] for row in pagerank_rows}


검색 점수와 PageRank로 후보를 재정렬하는 함수를 만듭니다.


In [ ]:
def minmax(values):
    """현재 후보 안에서 값을 0~1로 변환합니다. 동점만 있으면 모두 0.5입니다."""
    low, high = min(values), max(values)
    # 모두 동점이면 0.5로 통일합니다.
    return [0.5 if high == low else (value - low) / (high - low) for value in values]


def rerank_candidates(rows, weight):
    """같은 후보에 RRF와 영화 PageRank를 결합합니다.

    Args:
        rows: 관계 조건을 통과한 통합 후보.
        weight: PageRank 비중. 0이면 RRF만 반영합니다.
    Returns:
        0~1로 바꾼 두 점수와 최종 점수(final_score)를 붙여 정렬한 목록.
    """
    # 조건을 통과한 후보가 없으면 재정렬 없이 빈 목록을 반환합니다.
    if not rows:
        return []
    # 같은 후보에서 RRF와 PageRank 점수를 각각 0~1로 바꿉니다.
    retrieval = minmax([row["rrf_score"] for row in rows])
    centrality = minmax([pagerank_by_id[row["source_id"]] for row in rows])
    ranked = []
    # 각 후보에 그 문서의 검색 점수와 PageRank 점수를 연결합니다.
    for row, relevance, importance in zip(rows, retrieval, centrality):
        # weight 비율만큼 PageRank를 반영해 새 점수를 만듭니다.
        ranked.append({**row, "rrf_norm": relevance, "pagerank_norm": importance,
                       "final_score": (1 - weight) * relevance + weight * importance})
    # 같은 후보를 결합 점수순으로 재정렬합니다.
    return sorted(ranked, key=lambda row: (-row["final_score"], row["source_id"]))


근거를 모으는 함수와 근거를 인용하는 답변 체인을 만듭니다.


In [ ]:
# claim_id로 출연 근거를 찾을 사전을 만듭니다.
relations_by_id = {row["claim_id"]: row for row in movies["relations"]}


def build_evidence(ranked, eligible_rows, top_n=3):
    """상위 영화의 줄거리·개봉연도·출연 근거를 모읍니다."""
    # 후보 원문 ID로 영화 속성과 출연 경로를 찾습니다.
    eligible_by_id = {row["source_id"]: row for row in eligible_rows}
    evidence = []
    claim_ids = set()

    # 상위 영화의 줄거리와 이미 조회한 제목·개봉연도를 담습니다.
    for row in ranked[:top_n]:
        source_id = row["source_id"]
        doc = documents_by_id[source_id]
        movie = eligible_by_id[source_id]
        evidence.extend([
            {"evidence_id": source_id, "kind": "plot", "title": doc["title"],
             "url": doc["url"], "text": doc["text"]},
            {"evidence_id": "movie:" + movie["movie_id"], "kind": "movie_record",
             "url": "https://github.com/neo4j-graph-examples/movies",
             "text": f"{movie['title']} / 개봉연도: {movie['released']}"},
        ])
        # 양쪽 출연 관계 ID를 집합에 모으면 중복이 자동으로 제거됩니다.
        for path in movie["paths"]:
            claim_ids.update([path["first_claim"], path["second_claim"]])

    # 모아 둔 관계 ID로 실제 근거 문장과 출처를 가져옵니다.
    for claim_id in sorted(claim_ids):
        claim = relations_by_id[claim_id]
        evidence.append({"evidence_id": claim_id, "kind": "cast_relation",
                         "url": claim["source_url"], "text": claim["evidence"]})
    return evidence


# 줄거리·연도·출연 근거를 인용해 답하도록 프롬프트를 만듭니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공한 영화 데이터만 사용해 한국어로 답하세요. 줄거리와 배우·연도 조건을 모두 확인하세요. "
               "답변은 세 항목으로 작성하세요. 1. 작품과 개봉연도, 2. 기준 영화와 공유하는 배우, 3. 질문에 맞는 줄거리. "
               "각 항목의 문장 바로 뒤에 실제 evidence_id를 인용하세요. 대괄호 하나에는 ID 하나만 넣으세요. 여러 ID는 [ID1][ID2] 형태로 붙이고 쉼표로 묶지 마세요. "
               "연도에는 movie_record, 줄거리에는 plot, 공유 배우마다 양쪽 cast_relation의 ID를 인용하세요. "
               "예: - 작품과 연도: 작품명과 연도 [해당 영화 속성 ID]. "
               "evidence_ids 목록에만 ID를 적는 것은 충분하지 않습니다. answer 본문에도 반드시 표시하세요. "
               "배우 본인이 사고를 겪었다고 쓰지 말고 배우와 극중 인물을 구분하세요. "
               "조건은 맞지만 줄거리가 다른 후보를 선택하지 마세요. 부족한 근거는 부족하다고 말하세요. "
               "evidence_ids에는 answer 본문에 인용한 모든 ID를 빠짐없이 한 번씩 담으세요. ID마다 대괄호를 따로 쓰세요. 질문의 줄거리 조건에 맞는 영화가 없으면 유사한 다른 영화를 정답으로 제시하지 마세요. 각 항목은 한 문장으로 쓰고, 줄거리는 질문의 조건에 해당하는 핵심 사건만 요약하세요."),
    ("human", "질문: {question}\n근거: {context}"),
])
# 답변 본문과 인용 ID 목록을 구조화합니다.
class GroundedAnswer(BaseModel):
    answer: str = Field(description="작품·연도, 공유 배우, 줄거리의 세 항목. 각 항목은 한 문장. 본문에 [실제 evidence_id]를 표시하며 대괄호 하나에 ID 하나만 넣은 한국어 답변")
    evidence_ids: list[str] = Field(description="answer 본문의 모든 대괄호 ID를 빠짐없이 담은 목록. 줄거리의 plot 원문 ID, 영화 속성의 movie_record ID, 공유 배우 양쪽 cast_relation ID를 모두 포함합니다.")


# 답변 체인을 구성합니다. invoke에서 실제 호출합니다.
answer_chain = answer_prompt | llm.with_structured_output(GroundedAnswer)


세 검색 결과를 통합하고 배우·연도 조건을 만족하는 영화만 남깁니다.


In [ ]:
question = "The Matrix와 배우를 공유하고 1999년 이전 개봉한 다른 영화 중, 뇌에 기밀 데이터를 저장·운반하다 위험에 빠지는 작품은 무엇인가요?"
semantic_query = "기밀 정보를 뇌 속 장치에 저장해 운반하는 사람이 위험에 빠지는 영화"
filters = {"anchor": "The Matrix", "min_year": 0, "max_year": 1998, "min_shared": 1}
graph_request = "매트릭스(The Matrix)와 배우를 최소 한 명 공유하며 1999년 이전에 개봉한 다른 영화를 모두 찾아 주세요. 원문 ID는 source_id, 제목은 title로 반환하세요."

# 질문 벡터와 관계 검색 결과를 한 번씩 구합니다.
query_vector = embedding_model.embed_query(semantic_query)
graph_result = graph_chain.invoke({"query": graph_request})
graph_rows = graph_result["result"]
# 배우·연도 조건에 맞는 영화와 양쪽 출연 근거를 조회합니다.
eligible_rows = run_cypher(eligible_query, **filters)
allowed_ids = {row["source_id"] for row in eligible_rows}
routes = {
    "fulltext": fulltext_documents(semantic_query, top_k=30),
    "dense": vector_documents(query_vector, top_k=30),
    "graph": graph_rows,
}
weights = {"fulltext": 0.35, "dense": 0.45, "graph": 0.20}
# 높은 점수를 받아도 배우·연도 조건에 맞지 않으면 제외합니다.
candidate_rows = [row for row in fuse_routes(routes, weights) if row["source_id"] in allowed_ids]


후보를 재정렬하고 근거를 인용한 답변을 생성합니다.


In [ ]:
# PageRank 비중 0.2로 정렬한 상위 3편의 근거를 모읍니다.
final_ranked = rerank_candidates(candidate_rows, weight=0.2)
evidence = build_evidence(final_ranked, eligible_rows, top_n=3)
answer = answer_chain.invoke({"question": question, "context": json.dumps(evidence, ensure_ascii=False)})
print(answer.answer)
